# Part 15 - Agent to Agent: A2A delegation across boundaries

*Agents from First Principles, Part 15.*

Part 14's handoff transfers control to a specialist that lives in YOUR process: you import it, you call it, the conversation moves to a colleague at the same desk. Real organizations are not one process. The billing agent is owned by another team, runs on another host, behind another deploy. You cannot import it. You need a way for one agent to DISCOVER and DELEGATE to another agent across an organizational boundary, and a way to decide whether to TRUST it.

That is what **A2A** (the Agent-to-Agent protocol) is for. Part 12 gave us MCP, which is **VERTICAL**: an agent reaching DOWN to its tools. A2A is **HORIZONTAL**: an agent reaching ACROSS to a peer agent. Same JSON-RPC plumbing (we reuse Part 12's transport), opposite direction.

This part builds the A2A essentials by hand, standard library only:

1. **Agent Cards.** A public description an agent serves at a well-known URL (`/.well-known/agent-card.json`): its name, what SKILLS it offers, the input/output modes it speaks, and how to AUTHENTICATE. Discovery is just fetching this card.
2. **Delegation over JSON-RPC.** The support agent sends the billing agent a task (`tasks/send`). A2A tasks have a LIFECYCLE: submitted to working to completed, with the result returned as an artifact. We reuse Part 12's in-process JSON-RPC shim, printing the literal frames.
3. **A tiny registry** so the support agent can look up where the billing agent lives.
4. **Trust and allowlist.** Discovering an agent is not trusting it. Before delegating, the support agent checks the peer against an ALLOWLIST. A peer that is not on the list is refused, and any output from outside the allowlist is tagged UNTRUSTED so a later step treats it as suspect. This stays DELIBERATELY THIN here: the full agentic-security treatment (injection through a delegated result, the lethal trifecta) is Part 16's job.

A2A traces render as the Part 11 span tree, the same as everything else (forward-ref).

> **Runs with no network, no API key, and no third-party dependency.** The in-process JSON-RPC shim is the offline source of truth; the real network transport is one flag away. Set `OPENAI_API_KEY` to see the real banner; the code always uses the deterministic shim so output stays reproducible.

Companion script: `a2a_delegation.py`

## Setup

Two standard-library imports do the work: `json` serializes every JSON-RPC frame (the exact bytes that would cross a socket, exactly as in Part 12), and `os` lets us check for an API key without ever requiring one. Everything else is plain Python, so every cell runs fully offline.

A note on the spec: the A2A field names and method strings move fast (the protocol is young). Check the current spec before going live. Only the transport and the protocol strings would ever need edits.

In [ ]:
import json
import os

print("ready -- no network, no API key, no third-party dependency required")

## Step 1 - The Agent Card: what a peer publishes about itself

An **Agent Card** is the A2A analogue of MCP's `tools/list`, but one level up: instead of describing the tools inside one agent, it describes a whole agent as a peer you might delegate to. A peer serves it at a well-known URL (`/.well-known/agent-card.json`), and it carries everything a caller needs before reaching out:

- **`name`** and **`description`** - who this is and what it does.
- **`url`** - where to send tasks (also the discovery address).
- **`skills`** - the named capabilities it offers, each with its input/output modes. This is what you delegate TO.
- **`authentication`** - which auth schemes it accepts.

`BILLING_CARD` is a vetted peer that issues refunds with `bearer` auth. `ROGUE_CARD` is a second peer that ALSO claims the `process_refund` skill but that we have never vetted, and it advertises `none` for auth. Both look plausible from the card alone. That is the whole point of Step 4: a card tells you what a peer CLAIMS, not whether you should believe it.

In [ ]:
# The Agent Card the billing peer publishes at /.well-known/agent-card.json.
# Discovery is fetching this document.
BILLING_CARD = {
    "name": "billing-agent",
    "description": "Issues refunds and credits for orders.",
    "url": "https://billing.acme.example/a2a",
    "version": "1.0.0",
    "skills": [{"id": "process_refund", "description": "Issue a refund for an order",
                "inputModes": ["application/json"], "outputModes": ["application/json"]}],
    "capabilities": {"streaming": False},
    "authentication": {"schemes": ["bearer"]},
}

# A second peer that ALSO claims to do refunds, but that we have never vetted.
ROGUE_CARD = {
    "name": "refund-bot-9000",
    "description": "Totally legit refunds, definitely.",
    "url": "https://refunds-r-us.example/a2a",
    "version": "0.0.1",
    "skills": [{"id": "process_refund", "description": "refunds!!",
                "inputModes": ["application/json"], "outputModes": ["application/json"]}],
    "capabilities": {"streaming": False},
    "authentication": {"schemes": ["none"]},
}

print("two agent cards defined: billing-agent (bearer), refund-bot-9000 (none).")

## Step 2 - A remote A2A agent: agent/getCard and tasks/send

A peer agent is reachable over JSON-RPC, the same transport Part 12's MCP used, pointed sideways instead of down. Its `handle()` dispatches two methods:

- **`agent/getCard`** serves the Agent Card. This is the discovery endpoint; fetching the card is how a caller learns the peer's skills and auth.
- **`tasks/send`** runs a named skill. Unlike a one-shot MCP `tools/call`, an A2A task has a LIFECYCLE: `submitted` to `working` to `completed`. In a real implementation the caller streams those state transitions; here we return them as a list so you can see the shape. The result of the task comes back as an **artifact** (here, a text artifact), not a bare return value.

The lifecycle is the conceptual difference from MCP. A tool call is synchronous and atomic; a delegated task is a unit of work with states you can observe, because the peer may take real time and report progress. An unknown skill returns a clean JSON-RPC error (`-32601`), in the spirit of Part 1's validator and Part 2's failure taxonomy: a peer never crashes the caller.

In [ ]:
def _billing_refund(order_id, amount):
    """The side-effecting refund, now owned by a PEER (Part 2's non-idempotent action)."""
    return f"refund of ${float(amount):.2f} issued for {order_id} (ref BILL-{order_id})"


class A2AAgent:
    """A remote peer reachable over JSON-RPC. agent/getCard serves the card;
    tasks/send runs a skill through the lifecycle submitted -> working -> completed
    and returns an artifact. (Reuses Part 12's in-process transport, pointed sideways.)"""

    def __init__(self, card, skill_fns):
        self.card = card
        self.skill_fns = skill_fns
        self._task = 0

    def handle(self, request):
        rid, method, params = request.get("id"), request["method"], request.get("params", {})
        if method == "agent/getCard":
            return {"jsonrpc": "2.0", "id": rid, "result": self.card}
        if method == "tasks/send":
            skill, args = params["skill"], params.get("input", {})
            if skill not in self.skill_fns:
                return {"jsonrpc": "2.0", "id": rid,
                        "error": {"code": -32601, "message": f"no skill '{skill}'"}}
            self._task += 1
            tid = f"task-{self._task}"
            # the lifecycle the caller would observe (streamed in a real impl):
            lifecycle = ["submitted", "working", "completed"]
            output = self.skill_fns[skill](**args)
            return {"jsonrpc": "2.0", "id": rid,
                    "result": {"id": tid, "lifecycle": lifecycle, "status": {"state": "completed"},
                               "artifacts": [{"type": "text", "text": output}]}}
        return {"jsonrpc": "2.0", "id": rid, "error": {"code": -32601, "message": "method not found"}}


print("A2AAgent ready (agent/getCard, tasks/send with a task lifecycle).")

## Step 3 - A tiny registry: where peers live

To delegate to a peer you first have to find it. A **registry** maps a well-known URL to the agent serving it, the smallest stand-in for service discovery (a real one would be a directory service, or just DNS plus the well-known path). `fetch_card(url)` is exactly the discovery call: it sends `agent/getCard` to whoever lives at that URL and returns the card. `agent_at(url)` hands back the peer object so the support agent can send it tasks.

This is deliberately thin. The point is only to separate "where is the peer" (the registry) from "what does it claim" (its card) from "do I trust it" (the allowlist in Step 4). Three different questions, three different mechanisms.

In [ ]:
class Registry:
    """Maps a well-known URL to the peer serving it. The smallest possible
    stand-in for service discovery."""

    def __init__(self):
        self._by_url = {}

    def publish(self, agent):
        self._by_url[agent.card["url"]] = agent

    def fetch_card(self, url):
        return self._by_url[url].handle(
            {"jsonrpc": "2.0", "id": 0, "method": "agent/getCard"})["result"]

    def agent_at(self, url):
        return self._by_url[url]


print("Registry ready (publish, fetch_card = discovery, agent_at).")

## Step 4 - The support agent: discover, check trust, then delegate

The support agent is the A2A CLIENT. It does three things:

- **`discover(url)`** fetches a peer's card from the registry and prints what it found (skills + auth). This is discovery, nothing more.
- **`delegate(url, skill, task_input)`** is where trust lives. Before sending a single frame, it checks the peer's name against an ALLOWLIST. A peer not on the list is REFUSED and `delegate` returns `None`; no task is ever sent. An allowlisted peer gets the `tasks/send` JSON-RPC call, and the support agent reads the lifecycle and the artifact off the response.
- **`_rpc()`** is the transport, lifted straight from Part 12's host: it prints every request and response frame verbatim with `json.dumps`, so the in-process shim shows the EXACT bytes a socket would carry. The frames flow sideways (agent to agent) instead of down (agent to tools), but they are the same JSON-RPC.

**Discovery is not trust.** This is the load-bearing line of the whole part. Fetching a card tells you what a peer claims; the allowlist decides whether to act on it. A result that came back from an allowlisted peer is labeled a trusted result; anything from outside the allowlist would be tagged UNTRUSTED so a later step treats it as suspect. We keep this THIN on purpose: the deep treatment (a malicious peer returning a prompt-injection payload in its artifact, the lethal trifecta) is Part 16.

In [ ]:
class SupportAgent:
    """The A2A client. Discovers a peer's card, checks it against an ALLOWLIST
    before trusting it, and delegates a task over JSON-RPC. Output from outside the
    allowlist is tagged UNTRUSTED (thin here; Part 16 goes deep)."""

    def __init__(self, registry, allowlist):
        self.registry = registry
        self.allowlist = set(allowlist)
        self._id = 0

    def _rpc(self, agent, method, params, show=True):
        self._id += 1
        req = {"jsonrpc": "2.0", "id": self._id, "method": method, "params": params}
        if show:
            print(f"    --> {json.dumps(req)}")     # real JSON-RPC frame (Part 12 transport)
        resp = agent.handle(req)                     # in-process shim: a direct call, real frames
        if show:
            print(f"    <-- {json.dumps(resp)}")
        return resp

    def discover(self, url):
        card = self.registry.fetch_card(url)
        print(f"    [support] discovered '{card['name']}' at {url}: "
              f"skills={[s['id'] for s in card['skills']]}, auth={card['authentication']['schemes']}")
        return card

    def delegate(self, url, skill, task_input):
        card = self.registry.fetch_card(url)
        if card["name"] not in self.allowlist:                       # discovery != trust
            print(f"    [trust] '{card['name']}' is NOT on the allowlist -> delegation REFUSED")
            return None
        print(f"    [trust] '{card['name']}' is allowlisted -> delegating")
        agent = self.registry.agent_at(url)
        resp = self._rpc(agent, "tasks/send", {"skill": skill, "input": task_input})
        task = resp["result"]
        artifact = task["artifacts"][0]["text"]
        print(f"    [support] task {task['id']} lifecycle: {' -> '.join(task['lifecycle'])}")
        print(f"    [support] trusted result (from {card['name']}): {artifact}")
        return artifact


print("SupportAgent ready (discover, delegate with the allowlist trust check).")

## Step 5 - generate(): the real LLM-driven path (reference shape only)

Same device as every prior part. On the real path the support agent is an LLM: you hand it the goal and the discovered peer cards, and it decides which peer to delegate to and with what input. `generate()` is the single swappable call that lights up that path; the offline demo never calls it, so output stays reproducible.

SDK names and model ids move fast; check current docs. Only `generate()` (and a real network transport in place of the in-process shim) would need edits to go live.

In [ ]:
def generate(prompt):
    """REAL path: a hosted LLM drives the support agent's delegation decisions. Unused offline."""
    from openai import OpenAI
    client = OpenAI()                               # reads OPENAI_API_KEY
    resp = client.chat.completions.create(
        model="gpt-4o-mini",                        # a small chat model; check names
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return resp.choices[0].message.content

# Anthropic / Claude alternative. Swap in for generate() above:
#
# def generate(prompt):
#     from anthropic import Anthropic
#     client = Anthropic()                            # reads ANTHROPIC_API_KEY
#     resp = client.messages.create(
#         model="claude-opus-4-8",                    # check current model names
#         max_tokens=1024,
#         messages=[{"role": "user", "content": prompt}],
#     )
#     return resp.content[0].text


if os.environ.get("OPENAI_API_KEY"):
    print("[support] OPENAI_API_KEY set; a real LLM would drive delegation. "
          "Falling through to deterministic logic for reproducibility.")
else:
    print("[support] no OPENAI_API_KEY; the support agent drives the in-process shim (offline default)")

## Demo - A2A on the wire

Everything below runs fully offline through the in-process JSON-RPC shim. The demo has four sections: (1) the Agent Card and discovery; (2) an allowlisted delegation, with its task lifecycle; (3) the trust boundary, where an unvetted peer is refused; (4) the MCP-vs-A2A summary.

We start with the banner and stand up the world: a registry holding both peers (the vetted billing agent and the rogue refund bot), and a support agent whose allowlist contains only `billing-agent`. The transport line states the framing this part shares with Part 12, and the direction it reverses.

In [ ]:
bar = "=" * 72
print(bar)
print("AGENT TO AGENT  -  A2A delegation across boundaries")
print(bar)
print("[transport] in-process JSON-RPC shim (Part 12 reused). MCP = vertical (agent -> tools);")
print("A2A = horizontal (agent -> agent). Same plumbing, opposite direction.")
if os.environ.get("OPENAI_API_KEY"):
    print("[support] OPENAI_API_KEY set; a real LLM would drive delegation. Falling through to "
          "deterministic logic for reproducibility.")

registry = Registry()
registry.publish(A2AAgent(BILLING_CARD, {"process_refund": _billing_refund}))
registry.publish(A2AAgent(ROGUE_CARD, {"process_refund": _billing_refund}))
support = SupportAgent(registry, allowlist={"billing-agent"})

print("\nworld ready: 2 peers published; support agent allowlist = {'billing-agent'}.")

### 1) The Agent Card and discovery

First, the billing agent's public card, the document it would serve at `/.well-known/agent-card.json`. A caller reads it before reaching out: the `skills` list says it can `process_refund`, the `authentication` block says it wants a `bearer` token. Then the support agent DISCOVERS it through the registry, the discovery call (`agent/getCard`) that fetches the card and reports the skills and auth it advertises. After this the support agent knows the peer exists and what it claims, but has not yet decided to trust it.

In [ ]:
print("\n" + "-" * 72)
print("1) AGENT CARD: the billing agent publishes one at /.well-known/agent-card.json")
print("-" * 72)
print(json.dumps(BILLING_CARD, indent=2))
print("\n  The support agent discovers it from the registry:")
support.discover(BILLING_CARD["url"])

### 2) Delegation over A2A JSON-RPC (the trust check passes)

Now the support agent delegates a refund. Because `billing-agent` is on the allowlist, the trust check passes and the task goes out. Watch the two JSON-RPC frames: the request is `tasks/send` (id 1) carrying the skill `process_refund` and the input `{order_id, amount}`; the response carries the task id, the lifecycle `["submitted", "working", "completed"]`, and the result as a text artifact. The support agent reads the lifecycle (the states it would have streamed in production) and the artifact, and labels it a trusted result because it came from an allowlisted peer. This is the horizontal counterpart of Part 12's `tools/call`, with a lifecycle wrapped around it.

In [ ]:
print("\n" + "-" * 72)
print("2) DELEGATION: support delegates a refund to the billing agent over JSON-RPC.")
print("-" * 72)
support.delegate(BILLING_CARD["url"], "process_refund",
                 {"order_id": "ORD-3300", "amount": 180.0})

### 3) Trust and allowlist: an unvetted peer is refused

Here is the load-bearing section. The rogue peer `refund-bot-9000` advertises the SAME `process_refund` skill, so from skills alone it looks interchangeable with the billing agent, and its card honestly admits `auth=['none']`. The support agent discovers it fine (discovery works for anyone), but when it tries to delegate, the allowlist check fails: `refund-bot-9000` is not on the list, so delegation is REFUSED before any task frame is sent, and the result is `None`.

This is the whole trust touchpoint of the part: **discovering a peer is not trusting it.** A card tells you what a peer claims; the allowlist decides whether to act on it. Any result from outside the allowlist is tagged UNTRUSTED so a later step treats it as suspect. We keep it thin here on purpose, the deep treatment (injection via a delegated result, the lethal trifecta) is Part 16.

In [ ]:
print("\n" + "-" * 72)
print("3) TRUST + ALLOWLIST: an unvetted peer claims the same skill.")
print("-" * 72)
support.discover(ROGUE_CARD["url"])
result = support.delegate(ROGUE_CARD["url"], "process_refund",
                          {"order_id": "ORD-3300", "amount": 180.0})
print(f"    -> delegation result: {result}  (refused; output never trusted)")
print("    Discovering a peer is not trusting it. Any result from outside the allowlist is")
print("    tagged UNTRUSTED. The deep treatment (injection via a delegated result, the lethal")
print("    trifecta) is Part 16.")

### 4) MCP vs A2A: two protocols, one transport

The closing frame for the part. MCP (Part 12) and A2A (this part) are the same JSON-RPC plumbing pointed in opposite directions. MCP is VERTICAL: an agent reaching DOWN to its tools, via `initialize` / `tools-list` / `tools-call`. A2A is HORIZONTAL: an agent reaching ACROSS to a peer agent, via `agent-card` / `tasks-send` plus a task lifecycle. Both ride JSON-RPC; what A2A adds on top is discovery via Agent Cards and a TRUST BOUNDARY between peers, because a peer is another party, not your own tool.

In [ ]:
print("\n" + bar)
print("MCP vs A2A: two protocols, one transport.")
print(bar)
print("  MCP (Part 12): agent -> TOOLS      (vertical)   initialize / tools-list / tools-call")
print("  A2A (this part): agent -> AGENT    (horizontal) agent-card / tasks-send + a task lifecycle")
print("  Both ride JSON-RPC; A2A adds discovery via Agent Cards and a trust boundary between peers.")

print("\n" + bar)
print("Done. Handoffs stop at the process boundary; A2A crosses it:")
print("  - an AGENT CARD at a well-known URL advertises a peer's skills + auth")
print("  - DELEGATION is a JSON-RPC task with a submitted -> working -> completed lifecycle")
print("  - a REGISTRY locates peers; an ALLOWLIST decides which to trust")
print("  - discovery is not trust: unvetted output is tagged untrusted (Part 16 goes deep)")
print(bar)

## Wrap-up: the agent crosses the boundary

Part 14's handoff moved control to a specialist at the same desk, in the same process. This part reached across the building. An agent can now find a peer it never imported, ask what it can do, delegate a task to it, and decide whether the answer is worth believing.

- **Agent Cards make peers discoverable.** A peer publishes a card at a well-known URL describing its name, skills, modes, and auth. Discovery is fetching the card, the horizontal cousin of MCP's `tools/list`.
- **Delegation is a JSON-RPC task with a lifecycle.** `tasks/send` runs a skill through `submitted` to `working` to `completed`, returning the result as an artifact, not a bare value. Same transport as Part 12, pointed sideways.
- **A registry locates peers; an allowlist decides which to trust.** Three separate questions: where a peer lives, what it claims, and whether to act on it.
- **Discovery is not trust.** An unvetted peer that claims the right skill is still refused, and any output from outside the allowlist is tagged UNTRUSTED. We kept this touchpoint thin on purpose.
- **MCP is vertical, A2A is horizontal.** Two protocols, one JSON-RPC transport; A2A adds Agent Cards and a trust boundary between peers.

**Part 16 - securing the agent.** We just admitted output from a peer with a single thin line and waved at the danger. Part 16 goes deep: what happens when a peer (or a retrieved document, or a tool result) carries a prompt-injection payload, the lethal trifecta of untrusted input plus private data plus the ability to act, and what it actually takes to defend an agent that talks to the outside world.